# 11 | Generalización espacial

> **Cuaderno de reproducción (posterior).** Generado el 2026-10-23 como parte de la entrega docente. El repositorio original no contenía cuadernos versionados; estos reconstruyen los análisis a partir de los scripts y módulos existentes.

**Tipo de ejecución: CONSULTA — lee resultados previamente generados; NO entrena modelos.**

In [1]:
import os
from pathlib import Path

def _raiz_repositorio() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / 'data' / 'raw' / 'interseccion_terrenos_zhg_r1_yacopi.csv').exists():
            return p
    raise FileNotFoundError('No se encontro la raiz del repositorio (busca data/raw)')

REPO = _raiz_repositorio()
print('Raiz del repositorio resuelta (carpeta):', REPO.name, '| data/raw encontrado')

Raiz del repositorio resuelta (carpeta): yacopi-project | data/raw encontrado


## LOZO (leave-one-zone-out)

Generado por `ml/src/yacopi_ml/valor/evaluate.py` y guardado en `desgloses_esc_A.json`. Cada zona se evalúa con un CatBoost entrenado sobre TODAS las demás zonas (la zona evaluada nunca aparece en su entrenamiento).

In [2]:
import json
d = json.loads((REPO / 'ml/artifacts/exp-valor-terreno-001/desgloses_esc_A.json').read_text(encoding='utf-8'))
print('MAE CatBoost dejando cada zona completa fuera (LOZO):')
for k, v in sorted(d['lozo_por_zona'].items()):
    print(f"  zona {k:>3s} n={v['n']:>4d} MAE={v['mae_cop_ha']:>10.0f} R2={v['r2']:+.3f}")
print('suma de n:', d['lozo_suma_n'])

MAE CatBoost dejando cada zona completa fuera (LOZO):
  zona   1 n=   8 MAE=   2692400 R2=+0.227
  zona  10 n= 836 MAE=    486053 R2=-0.461
  zona  11 n= 138 MAE=    174866 R2=-7.138
  zona   2 n= 143 MAE=    977995 R2=-8.868
  zona   3 n= 567 MAE=   1192263 R2=+0.385
  zona   4 n= 288 MAE=    582650 R2=+0.716
  zona   5 n= 301 MAE=    210487 R2=-2.827
  zona   6 n= 235 MAE=    565477 R2=+0.509
  zona   7 n= 874 MAE=    263575 R2=-11.292
  zona   8 n= 122 MAE=    504095 R2=+0.614
  zona   9 n=1164 MAE=    563869 R2=+0.031
suma de n: 4676


## Por qué antes aparecían 4.668 registros (y no 4.676)

El LOZO original excluía zonas con menos de 30 predios: la zona 1 tiene **8 predios** y quedaba fuera (4.676 − 8 = 4.668). En la revisión de cierre se eliminó ese umbral: ahora se evalúan las 11 zonas y la suma coincide con los 4.676 predios del escenario A.

In [3]:
import sys
sys.path.insert(0, str(REPO / 'ml' / 'src'))
from yacopi_ml.valor.dataset import construir_dataset
ds = construir_dataset('A')
print('zona 1 en escenario A:', int((ds.codigo_zona_dominante == '1').sum()), 'predios')
print('zonas en A:', ds.codigo_zona_dominante.unique().tolist())

zona 1 en escenario A: 0 predios
zonas en A: [10, 11, 4, 5, 9, 7, 2, 8, 6, 3, 1]


In [4]:
print('Verificacion de independencia LOZO (por construccion, evalua evaluate.py):')
for zona, g in ds.groupby('codigo_zona_dominante'):
    tr_z = ds[ds.codigo_zona_dominante != zona]
    assert not set(tr_z.numero_predial) & set(g.numero_predial)
print('  OK: en ninguna zona hay solapamiento con su entrenamiento')

Verificacion de independencia LOZO (por construccion, evalua evaluate.py):
  OK: en ninguna zona hay solapamiento con su entrenamiento


## Lectura

El desempeño depende de la partición: test predio (aleatorio) vs test espacial (bloques) vs LOZO (zonas fuera). La buena métrica en test predio NO garantiza generalización a zonas no vistas.